# UFC Fight Predictor — Step 1: Load & Clean
Goal of this notebook: get `UFC_full_data_silver_v2.csv` into a trustworthy shape **before** any feature engineering or modeling.

Folder layout expected:
```
your-project/
  data/UFC_full_data_silver_v2.csv
  01_load_and_clean.ipynb   <- this file
```

In [9]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

df = pd.read_csv("data/UFC_full_data_silver_v2.csv", parse_dates=["event_date"])
print(df.shape)
print(df.event_date.min().date(), "->", df.event_date.max().date())

(8937, 361)
1994-03-11 -> 2026-10-10


## 1. Drop fights with no winner
Overturned results, no-contests, and decisions with no recorded winner can't be used as win/loss labels.

In [10]:
print("Fights with no winner:", df.winner.isna().sum())
print(df.loc[df.winner.isna(), "result"].value_counts(), "\n")

df = df[df.winner.notna()].copy()
print("Fights remaining:", len(df))

Fights with no winner: 157
result
Overturned              59
Decision - Majority     39
Could Not Continue      33
Decision - Split        17
Decision - Unanimous     7
Other                    2
Name: count, dtype: int64 

Fights remaining: 8780


## 2. Problem found: fighter 1 ALWAYS wins
In the raw file `f_1` is the winner in every row. A model trained on this would just learn "pick fighter 1" and score a fake 100%.
Let's prove it, then fix it.

In [11]:
print("Share of fights where winner == f_1:", (df.winner == df.f_1_name).mean())
print("Share of fights where winner == f_2:", (df.winner == df.f_2_name).mean())

Share of fights where winner == f_1: 1.0
Share of fights where winner == f_2: 0.0


**Fix:** for a random 50% of fights, swap *every* `f_1_*` column with its `f_2_*` partner. After the swap, "fighter A" (slot 1) wins about half the time, and the label is `a_wins`.

In [12]:
f1_cols = [c for c in df.columns if c.startswith("f_1_")]
f2_cols = [c.replace("f_1_", "f_2_", 1) for c in f1_cols]
assert all(c in df.columns for c in f2_cols), "Some f_1 columns have no f_2 partner"

swap_map = {**dict(zip(f1_cols, f2_cols)), **dict(zip(f2_cols, f1_cols))}

rng = np.random.default_rng(42)          # fixed seed = reproducible
flip = rng.random(len(df)) < 0.5

flipped = df[flip].rename(columns=swap_map)[df.columns]   # rename swaps both directions at once
df = (pd.concat([df[~flip], flipped])
        .sort_values("event_date", kind="stable")
        .reset_index(drop=True))

df["a_wins"] = (df.winner == df.f_1_name).astype(int)

In [13]:
# Sanity checks
print("a_wins mean (should be ~0.5):", round(df.a_wins.mean(), 3))
assert ((df.winner == df.f_1_name) | (df.winner == df.f_2_name)).all()
assert (df.a_wins == (df.winner == df.f_1_name)).all()
df[["event_date", "f_1_name", "f_2_name", "winner", "a_wins"]].sample(8, random_state=1)

a_wins mean (should be ~0.5): 0.497


,event_date,f_1_name,f_2_name,winner,a_wins
2975,2014-12-13,Rafael Dos Anjos,Nate Diaz,Rafael Dos Anjos,1
1005,2008-11-15,Jorge Gurgel,Aaron Riley,Aaron Riley,0
8647,2026-08-01,Jan Blachowicz,Navajo Stirling,Navajo Stirling,0
7308,2023-12-16,Cody Garbrandt,Brian Kelleher,Cody Garbrandt,1
7130,2023-08-05,Kennedy Nzechukwu,Dustin Jacoby,Dustin Jacoby,0
3423,2015-12-10,Danny Roberts,Nathan Coy,Danny Roberts,1
3961,2017-01-28,Jordan Johnson,Henrique da Silva,Jordan Johnson,1
2726,2014-06-28,Robert Whittaker,Mike Rhodes,Robert Whittaker,1


## 3. Proof that the career stats are snapshots (leakage check)
The columns like `f_1_fighter_w` (career wins) and `f_1_fighter_SlpM` come from *today's* fighter page, not from the day of the fight.
If they were "as of the fight", a veteran's numbers would change from fight to fight. Let's check.

In [14]:
stat_cols = ["fighter_f_name", "fighter_l_name", "fighter_w", "fighter_l", "fighter_SlpM", "fighter_TD_Def"]
a = df[["event_date"] + [f"f_1_{c}" for c in stat_cols]]
b = df[["event_date"] + [f"f_2_{c}" for c in stat_cols]]
a.columns = b.columns = ["event_date"] + stat_cols
long = pd.concat([a, b])
long["name"] = long.fighter_f_name.fillna("") + " " + long.fighter_l_name.fillna("")

g = long.groupby("name").agg(n_fights=("event_date", "size"),
                              unique_career_wins=("fighter_w", "nunique"),
                              unique_slpm=("fighter_SlpM", "nunique"))
vets = g[g.n_fights >= 8]
print("Fighters with 8+ fights:", len(vets))
print("Share whose career-wins value NEVER changes across fights:", round((vets.unique_career_wins == 1).mean(), 2))
print("Share whose SlpM value NEVER changes across fights:", round((vets.unique_slpm == 1).mean(), 2))

# One concrete example
example = long[long.name == "Jim Miller"].sort_values("event_date")
example[["event_date", "name", "fighter_w", "fighter_l", "fighter_SlpM"]].iloc[[0, len(example)//2, -1]]

Fighters with 8+ fights: 802
Share whose career-wins value NEVER changes across fights: 0.92
Share whose SlpM value NEVER changes across fights: 0.96


,event_date,name,fighter_w,fighter_l,fighter_SlpM
980,2008-10-18,Jim Miller,39,19,2.92
3845,2016-11-12,Jim Miller,39,19,2.92
8522,2026-05-09,Jim Miller,39,19,2.92


**Conclusion:** do NOT use `*_fighter_w/l/d`, `SlpM`, `Str_Acc`, `SApM`, `Str_Def`, `TD_*`, `Sub_Avg` as model inputs.
Next notebook: rebuild each fighter's stats **using only their earlier fights** (rolling averages, win streaks, days since last fight).

Safe to use as-is: height, reach, stance, DOB (-> age at fight), weight class, title fight, rounds scheduled.

## 4. Missing values in the columns we plan to keep

In [15]:
keep = ["f_1_fighter_height_cm", "f_1_fighter_reach_cm", "f_1_fighter_stance", "f_1_fighter_dob",
        "f_2_fighter_height_cm", "f_2_fighter_reach_cm", "f_2_fighter_stance", "f_2_fighter_dob",
        "weight_class", "title_fight", "num_rounds"]
df[keep].isna().mean().round(3).sort_values(ascending=False)

f_1_fighter_reach_cm     0.074
f_2_fighter_reach_cm     0.074
f_2_fighter_dob          0.009
f_1_fighter_dob          0.007
f_1_fighter_stance       0.006
f_2_fighter_stance       0.005
num_rounds               0.004
f_2_fighter_height_cm    0.002
f_1_fighter_height_cm    0.001
weight_class             0.000
title_fight              0.000
dtype: float64

## 5. Save the cleaned file

In [16]:
import os
os.makedirs("data", exist_ok=True)
df.to_csv("data/ufc_clean.csv", index=False)
print("Saved", df.shape, "-> data/ufc_clean.csv")

Saved (8780, 362) -> data/ufc_clean.csv
